In [11]:
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt

RAW_PATH = '../02_Data/'
PROCESSED_PATH = '../02_Data/processed'
os.makedirs(PROCESSED_PATH, exist_ok=True)

def clean_price(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r'[\$,€£]', '', regex=True).str.strip() , errors='coerce'
    )

def snapshot(df, name):
    return {
        'dataset' : name,
        'rows' : df.shape[0],
        'cols': df.shape[1],
        'columns': list(df.columns),
        'dtypes': df.dtypes.astype(str).to_dict()
    }


In [12]:
listings = pd.read_csv('../02_Data/listings.csv')

calendar_raw = pd.read_csv(os.path.join(RAW_PATH, 'calendar.csv'))
calendar = calendar_raw.copy()

calendar['dates'] = pd.to_datetime(calendar['date'], errors='coerce')
print(f"Invalid dates: {calendar['date'].isnull().sum()}")

# Availability flag (t/f → bool)
calendar['available'] = calendar['available'].map({'t': True, 'y': False})

# Minimum/maximum nights (numeric)
for col in ['minimum_nights', 'maximum_nights']:
    if col in calendar.columns:
        calendar[col] = pd.to_numeric(calendar[col], errors='coerce')

print(f"\n Calendar cleaned: {calendar.shape}")

rows_per_listing = calendar.groupby('listing_id').size()
print("Rows per listing:")
print(rows_per_listing.describe())

# Uniqueness check on (listing_id, date)
dups = calendar.duplicated(subset=['listing_id', 'date']).sum()
print(f"\nDuplicate (listing_id, date) pairs: {dups}")

calendar_orphans = calendar[~calendar['listing_id'].isin(listings['id'])]
print(f"Calendar listing_ids NOT in listings: {len(calendar_orphans):,}")

# Future vs. historical
print(f"\nCalendar date range: {calendar['date'].min()} → {calendar['date'].max()}")


Invalid dates: 0

 Calendar cleaned: (5623190, 6)
Rows per listing:
count    15406.0
mean       365.0
std          0.0
min        365.0
25%        365.0
50%        365.0
75%        365.0
max        365.0
dtype: float64

Duplicate (listing_id, date) pairs: 0
Calendar listing_ids NOT in listings: 41,245

Calendar date range: 2026-06-24 → 2027-07-02


In [13]:
calendar.to_csv(os.path.join(PROCESSED_PATH, 'calendar_clean.csv'), index=False)
print(f" Saved calendar_clean.csv  ({calendar.shape[0]:,} rows)")

 Saved calendar_clean.csv  (5,623,190 rows)
